# Zazo Brain — تدريب LoRA ثم رفعه إلى Hugging Face

هذا الدفتر يدرب محول LoRA لنموذج `google/gemma-2b-it`، ثم يرفع ملفي المحول إلى مستودع Hugging Face. بعد الرفع يمكنه اختيارياً تشغيل GitHub Action الموجود في مستودع Zazo ليقوم بالمسار التالي:

`Hugging Face → GitHub Action → Cloudflare Workers AI → active-lora في KV`

لا تضع أي Token داخل الدفتر. استخدم Colab Secrets مع تفعيل **Notebook access**. يلزم GPU في Colab، ويلزم توكن Hugging Face بصلاحية كتابة على مستودع المحول، مع قبول شروط Gemma أولاً.

## 1. اختيار ملف التدريب من الهاتف

شغّل الخلية التالية، ثم اضغط زر **Choose Files / اختيار الملفات** الذي سيظهر لفتح مجلدات الهاتف واختيار ملف التدريب.


In [ ]:
from pathlib import Path
from google.colab import files, drive

# سيظهر منتقي الملفات على الهاتف. اختر ملف JSONL أو JSON أو CSV أو TSV.
print('اختر ملف التدريب من الهاتف عندما تظهر نافذة الملفات...')
uploaded = files.upload()
if uploaded:
    name, content = next(iter(uploaded.items()))
    TRAIN_FILE = Path('/content') / Path(name).name
    TRAIN_FILE.write_bytes(content)
    print('تم رفع الملف من الهاتف:', TRAIN_FILE)
else:
    # إذا ألغيت الاختيار، نبحث اختيارياً داخل Google Drive.
    print('لم يتم اختيار ملف من الهاتف؛ سيتم البحث داخل Google Drive...')
    drive.mount('/content/drive', force_remount=False)
    DRIVE_INPUT = Path('/content/drive/MyDrive/zazo_brain_lora/input')
    SUPPORTED = {'.jsonl', '.ndjson', '.json', '.csv', '.tsv'}
    candidates = [p for p in DRIVE_INPUT.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED] if DRIVE_INPUT.exists() else []
    if not candidates:
        raise RuntimeError('لم يتم اختيار ملف من الهاتف ولم يوجد ملف تدريب في Google Drive.')
    TRAIN_FILE = max(candidates, key=lambda p: p.stat().st_mtime)
    print('تم اختيار أحدث ملف من Google Drive:', TRAIN_FILE)

print('ملف التدريب النهائي:', TRAIN_FILE)


## 1. فحص الموارد المتاحة في جلسة Colab

هذه الخلية توضح الموارد التي خصصتها Google لجلسة Runtime الحالية. لا يمكن للدفتر معرفة رصيد GPU على الحساب قبل تشغيل جلسة؛ اضغط **Connect** أو اختر GPU من **Runtime → Change runtime type** ثم شغّلها.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
try:
    import psutil
except ImportError:
    psutil = None

def gib(value):
    return round(value / (1024 ** 3), 2)

print('Python:', sys.version.split()[0])
print('Colab:', 'نعم' if 'google.colab' in sys.modules or Path('/content').exists() else 'غير مؤكد')

try:
    import torch
    gpu_count = torch.cuda.device_count()
    if not torch.cuda.is_available() or gpu_count == 0:
        print('GPU: غير متاح في جلسة Runtime الحالية')
        print('الحل: Runtime → Change runtime type → اختر T4 GPU أو GPU متاح → Save ثم أعد تشغيل الخلية.')
    else:
        print(f'GPU: متاح ({gpu_count} جهاز)')
        for index in range(gpu_count):
            props = torch.cuda.get_device_properties(index)
            free, total = torch.cuda.mem_get_info(index)
            print(f'  GPU {index}: {props.name}')
            print(f'  الذاكرة: {gib(total)} GiB إجمالي، {gib(free)} GiB متاح')
        try:
            smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,utilization.gpu', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False)
            if smi.stdout.strip(): print('nvidia-smi:\n' + smi.stdout.strip())
        except FileNotFoundError:
            pass
except Exception as error:
    print('تعذر فحص GPU:', error)

if psutil:
    memory = psutil.virtual_memory()
    print(f'RAM: {gib(memory.total)} GiB إجمالي، {gib(memory.available)} GiB متاح')
else:
    print('RAM: تعذر الفحص؛ psutil غير مثبتة بعد')
disk = shutil.disk_usage('/content')
print(f'مساحة /content: {gib(disk.total)} GiB إجمالي، {gib(disk.free)} GiB متاح')

if 'torch' in globals() and not torch.cuda.is_available():
    raise RuntimeError('أوقفنا الدفتر لأن GPU غير متاح. فعّل GPU ثم أعد تشغيل الخلية قبل التدريب.')

## 1. تثبيت الحزم

In [ ]:
%pip install -q -U transformers datasets peft accelerate bitsandbytes huggingface_hub safetensors requests
print('تم تثبيت الحزم')

## 2. الإعداد والتحقق من GPU

In [ ]:
from pathlib import Path
import os, json, csv, shutil, zipfile, getpass, re
import torch

if not torch.cuda.is_available():
    raise RuntimeError('فعّل GPU من Runtime → Change runtime type → T4 GPU أو أي GPU متاح قبل التدريب.')
print('GPU:', torch.cuda.get_device_name(0))

MODEL_NAME = 'google/gemma-2b-it'
HF_REPO_ID = 'zazo300/zazo-gemma-lora'
HF_REVISION = 'main'
MAX_LENGTH = 512
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
OUTPUT_DIR = Path('/content/zazo_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('مستودع Hugging Face:', HF_REPO_ID)

In [ ]:
from datasets import Dataset

def load_rows(path):
    text = path.read_text(encoding='utf-8-sig')
    suffix = path.suffix.lower()
    if suffix in {'.jsonl', '.ndjson'}:
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == '.json':
        value = json.loads(text)
        return value if isinstance(value, list) else value.get('records', value.get('data', []))
    delimiter = '\t' if suffix == '.tsv' else ','
    return list(csv.DictReader(text.splitlines(), delimiter=delimiter))

def row_to_text(row):
    if isinstance(row, dict) and isinstance(row.get('messages'), list):
        parts = []
        for message in row['messages']:
            role = message.get('role', 'user')
            content = str(message.get('content', '')).strip()
            if content:
                parts.append(f'{role}: {content}')
        return '\n'.join(parts)
    instruction = row.get('instruction', row.get('question', row.get('prompt', row.get('input', ''))))
    response = row.get('response', row.get('answer', row.get('output', '')))
    instruction, response = str(instruction or '').strip(), str(response or '').strip()
    return f'user: {instruction}\nassistant: {response}' if instruction and response else ''

rows = load_rows(TRAIN_FILE)
texts = [{'text': row_to_text(row)} for row in rows if isinstance(row, dict) and row_to_text(row)]
if not texts:
    raise RuntimeError('لم توجد سجلات صالحة. استخدم instruction/response أو question/answer أو messages.')
dataset = Dataset.from_list(texts)
print('عدد أمثلة التدريب:', len(dataset))
print(dataset[0]['text'][:1000])

## 4. تسجيل الدخول إلى Hugging Face

أنشئ Secret باسم `HF_TOKEN` بصلاحية **Write** على الأقل. لا تكتب قيمة التوكن داخل الخلية.

In [ ]:
from huggingface_hub import login, HfApi
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF_TOKEN من Hugging Face: ').strip()
if not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('توكن Hugging Face غير صالح.')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True, private=False)
print('تم التحقق من مستودع Hugging Face:', HF_REPO_ID)

## 5. تدريب Gemma 2B باستخدام LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto')
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], task_type=TaskType.CAUSAL_LM, bias='none'))
model.print_trainable_parameters()

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
args = TrainingArguments(output_dir=str(OUTPUT_DIR), num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION, learning_rate=2e-4, logging_steps=1, save_strategy='no', fp16=True, report_to='none', optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()
model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)
print('اكتمل التدريب:', OUTPUT_DIR)

## 6. رفع ملفات LoRA بالأسماء التي ينتظرها GitHub Action

In [ ]:
config_path = OUTPUT_DIR / 'adapter_config.json'
weights_path = OUTPUT_DIR / 'adapter_model.safetensors'
if not config_path.exists() or not weights_path.exists():
    raise FileNotFoundError('لم ينتج التدريب adapter_config.json و adapter_model.safetensors.')
config = json.loads(config_path.read_text())
config.setdefault('model_type', 'gemma')
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

manifest = {
    'base_model': MODEL_NAME,
    'examples': len(dataset),
    'trained_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
    'workflow': 'sync-huggingface-cloudflare.yml'
}
manifest_path = OUTPUT_DIR / 'training_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

api.upload_file(path_or_fileobj=str(config_path), path_in_repo='adapter_config (1).json', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload Zazo LoRA adapter config')
api.upload_file(path_or_fileobj=str(weights_path), path_in_repo='adapter_model (1).safetensors', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload Zazo LoRA adapter weights')
api.upload_file(path_or_fileobj=str(manifest_path), path_in_repo='training_manifest.json', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload training manifest')
print('تم رفع المحول إلى Hugging Face:', f'https://huggingface.co/{HF_REPO_ID}')

## 7. تشغيل GitHub Action اختيارياً

أنشئ Secret في Colab باسم `GH_TOKEN` بصلاحية Actions: write على مستودع GitHub. إذا لم تضفه، شغّل الـworkflow يدوياً من GitHub بعد انتهاء الخلية.

In [ ]:
import requests
GITHUB_REPO = 'zazotona301-oss/zazo'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

workflow_url = f'https://github.com/{GITHUB_REPO}/actions/workflows/sync-huggingface-cloudflare.yml'
if not GH_TOKEN:
    print('لم يتم تشغيل GitHub Action تلقائياً.')
    print('شغّله يدوياً من:', workflow_url)
else:
    response = requests.post(f'https://api.github.com/repos/{GITHUB_REPO}/actions/workflows/sync-huggingface-cloudflare.yml/dispatches', headers={'Accept': 'application/vnd.github+json', 'Authorization': f'Bearer {GH_TOKEN}', 'X-GitHub-Api-Version': '2022-11-28'}, json={'ref': 'main'}, timeout=30)
    if response.status_code != 204:
        raise RuntimeError(f'فشل تشغيل GitHub Action: HTTP {response.status_code} {response.text[:500]}')
    print('تم تشغيل GitHub Action بنجاح.')
    print('تابع النتيجة من:', workflow_url)

## النتيجة المتوقعة

بعد نجاح الـworkflow سيقوم GitHub بتحميل الملفين من Hugging Face، إنشاء fine-tune جديد في Cloudflare Workers AI، رفع الأوزان، ثم تحديث مفتاح `active-lora` في KV. لا تضع `CF_AI_TOKEN` أو `CF_KV_TOKEN` داخل Colab؛ هذه الأسرار تبقى في GitHub Actions Secrets كما هي حالياً.